In [1]:
import numpy as np
import networkx as nx
import pandas as pd
import pickle
import random
from sklearn.model_selection import train_test_split



MIMIC_resources = '/lustre/home/almusawiaf/PhD_Projects/MIMIC_resources'

def read_df(myPath):
    return pd.read_csv(myPath)

def get_dictionary(df, id1, id2):
    grouped_df = df.groupby(id1)[id2].agg(list)
    return grouped_df.to_dict()

def converting_dictionary(D, ch):
    '''reduce the icd code according to the type'''
    if ch == 'D':
        return {d: [f'{ch}_{str(e)[:3]}' for e in D[d]] for d in D}
    elif ch == 'R':
        return {d: [f'{ch}_{str(e)[:2]}' for e in D[d]] for d in D}
    elif ch =='M':
        return {d: [f'{ch}_{e}' for e in D[d]] for d in D}
    

def save_dict_to_pickle(dictionary, filename):
    import os
    print(f'Saving the dictionary to {filename}...')
    # Extract the directory path from the filename
    directory = os.path.dirname(filename)
    
    # Create the directory if it doesn't exist
    if not os.path.exists(directory):
        os.makedirs(directory)

    # Save the dictionary to the pickle file
    with open(filename, 'wb') as file:
        pickle.dump(dictionary, file)
    print('Saving complete...')

def load_dict_from_pickle(filename):
    with open(filename, 'rb') as file:
        loaded_dict = pickle.load(file)
    return loaded_dict

In [2]:
df_admission  = read_df(f'{MIMIC_resources}/ADMISSIONS.csv')
df_diagnosis  = read_df(f'{MIMIC_resources}/DIAGNOSES_ICD.csv')
df_patients   = read_df(f'{MIMIC_resources}/PATIENTS.csv')
df_medication = read_df(f'{MIMIC_resources}/PRESCRIPTIONS.csv')
df_procedures = read_df(f'{MIMIC_resources}/PROCEDURES_ICD.csv')


# task two: extracting the basic information

# removing empty admissions
df_admission.dropna(subset=['HADM_ID'], inplace=True)
df_admission.dropna(subset=['SUBJECT_ID'], inplace=True)
df_medication.dropna(subset=['drug'], inplace=True)
df_procedures.dropna(subset=['ICD9_CODE'])

# extracting unique values...
admissions = df_admission['HADM_ID'].unique()
patients = df_admission['SUBJECT_ID'].unique()
medications = df_medication['drug'].unique()
procedures = df_procedures['ICD9_CODE'].unique()
diagnoses = df_diagnosis['ICD9_CODE'].unique()

print(f'Patients \t= {len(patients)}\nAdmissions \t= {len(admissions)}\nMedications \t= {len(medications)}\nProcedures \t= {len(procedures)}')
print(len(diagnoses))

Patients 	= 46520
Admissions 	= 58976
Medications 	= 592
Procedures 	= 2009
6985


In [3]:
saving_path = f'Data'

# Originial order...

patients    = [f'P_{p}' for p in patients]
medications = [f'M_{p}' for p in medications]
procedures  = [f'R_{r}' for r in procedures]
diagnoses   = [f'D_{d}' for d in diagnoses]

In [4]:
# Processing the original data <complete dataset>..

# get a dictionary of {patient: [list of admissions]} and so on for the rest...
patient_admissions    = get_dictionary(df_admission, 'SUBJECT_ID', 'HADM_ID')

# get a dictionary of {admission: [list of items]} and so on for the rest...
admission_medications0 = get_dictionary(df_medication, 'hadm_id', 'drug')
admission_procedures0  = get_dictionary(df_procedures, 'HADM_ID', 'ICD9_CODE')
admission_diagnoses0   = get_dictionary(df_diagnosis, 'HADM_ID', 'ICD9_CODE')

# add the identifying letter to the code.
admission_medications = converting_dictionary(admission_medications0, 'M')
admission_diagnoses   = converting_dictionary(admission_diagnoses0, 'D')
admission_procedures  = converting_dictionary(admission_procedures0, 'R')

del admission_medications0
del admission_diagnoses0
del admission_procedures0

In [5]:
# Generating a dictionary of {admission: patient}
admission_patient = {}
for patient in patient_admissions:
    for adm in patient_admissions[patient]:
        admission_patient[adm] = patient

In [15]:
modified_admission_diagnoses = {}
for adm in admission_diagnoses:
    temp = admission_diagnoses[adm]
    new_Diagnoses = []
    for d in temp:
        if d not in new_Diagnoses:
            new_Diagnoses.append(d)
    modified_admission_diagnoses[adm] = new_Diagnoses
    

In [16]:
D = admission_diagnoses[100001]
new_Diagnoses = []
for d in D:
    if d not in new_Diagnoses:
        new_Diagnoses.append(d)
print(D)
print(new_Diagnoses)
print(list(set(D)))

['D_250', 'D_337', 'D_584', 'D_578', 'D_V58', 'D_250', 'D_536', 'D_458', 'D_250', 'D_403', 'D_585', 'D_250', 'D_362', 'D_250', 'D_707', 'D_V13']
['D_250', 'D_337', 'D_584', 'D_578', 'D_V58', 'D_536', 'D_458', 'D_403', 'D_585', 'D_362', 'D_707', 'D_V13']
['D_536', 'D_403', 'D_585', 'D_337', 'D_250', 'D_V13', 'D_V58', 'D_584', 'D_362', 'D_578', 'D_707', 'D_458']


In [20]:
# save_dict_to_pickle(patient_admissions,  'patient_admissions.pkl')
save_dict_to_pickle(modified_admission_diagnoses, '/lustre/home/almusawiaf/PhD_Projects/Semi_Supervised/Data/admission_diagnoses.pkl')
# save_dict_to_pickle(admission_procedures, 'admission_procedures.pkl')
# save_dict_to_pickle(admission_medications, 'admission_medications.pkl')

Saving the dictionary to /lustre/home/almusawiaf/PhD_Projects/Semi_Supervised/Data/admission_diagnoses.pkl...
Saving complete...
